In [0]:
%sql 
CREATE OR REPLACE TABLE taxi_labs.new_data.taxi_data_quarantined
TBLPROPERTIES ('delta.feature.timestampNtz' = 'supported')
as
select *, concat_ws(',',
    CASE WHEN NOT COALESCE(fare_amount >= 0, FALSE) THEN 'valid_fare' END,
    CASE WHEN NOT COALESCE(trip_distance BETWEEN 0 AND 500, FALSE) THEN 'valid_distance' END,
    CASE WHEN NOT COALESCE(tpep_pickup_datetime < tpep_dropoff_datetime, FALSE) THEN 'valid_times' END
) AS failed_rules
from (select distinct * from taxi_labs.new_data.new_bronze_taxi)
WHERE NOT (coalesce(fare_amount >= 0, false)
       AND coalesce(trip_distance BETWEEN 0 AND 500, false)
       AND coalesce(tpep_dropoff_datetime >= tpep_pickup_datetime, false)); 